# Blackbody spectral radiance

Compare thermal spectra at three temperatures with wavelength on one array
axis and temperature on the other. This is the notebook version of the
README example. Run the setup in [examples/README.md](README.md) first.

For an ideal blackbody, the spectral radiance per unit wavelength is
$$
B_\lambda(T)=\frac{2hc^2}{\lambda^5}
\left[\exp\!\left(\frac{hc}{\lambda k_B T}\right)-1\right]^{-1}.
$$
Here $h$ is Planck's constant, $c$ is the speed of light, $k_B$ is Boltzmann's
constant, and $T$ is absolute temperature. The exponential argument is
dimensionless. Radiance is per unit solid angle; it is not the hemispheric
radiant exitance, which has an additional factor of $\pi$ for this model.

Reference: [CIE, Planck's law](https://cie.co.at/eilvterm/17-24-005).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from unitx.si import SI, si

## Broadcast over wavelengths and temperatures

The wavelength array has shape `(100, 1)` and the temperature array `(3,)`.
Each expression below produces a `(100, 3)` Quantity with one uniform unit.
`expm1` evaluates $\exp(x)-1$ accurately near zero. The selected range also
keeps the short-wavelength exponential within floating-point range.

In [ ]:
lam = np.linspace(0.01, 3, 100)[:, None] * si.um  # wavelengths in microns
T = np.asarray([3000, 4000, 5000]) * si.K  # temperatures in Kelvin

nu = SI.c / lam
radiance = 2 * SI.h * SI.c**2 / (lam**5 * np.expm1(SI.h * nu / (SI.kB * T)))
radiance = radiance.to('kW/sr/m2/nm')  # spectral radiance per unit wavelength

print('Radiance shape:', radiance.shape)
print('Radiance unit:', radiance.unit)

## Plot with explicit numerical exits

Matplotlib receives values in the units written on the axes. The spectral
density conversion includes the change from per metre to per nanometre.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
ax.plot(lam.value, radiance.value)
ax.set(
    xlabel=f'Wavelength [{lam.unit}]',
    ylabel=f'Spectral radiance [{radiance.unit}]',
    title='Blackbody radiation',
)
ax.legend([f'T = {temperature}' for temperature in T])
ax.grid(True)
plt.show()

## Check the wavelength peaks

Wien's displacement law gives $\lambda_{\max}T=b$, with `SI.b` provided by
unitx. A grid maximum has finite wavelength resolution; compare it to the
continuous prediction with an absolute tolerance tied to the grid spacing.
This is the peak of $B_\lambda$, not the peak of $B_\nu$ mapped to wavelength.

In [ ]:
assert radiance.shape == (100, 3)
assert np.all(np.isfinite(radiance.value)) and np.all(radiance.value >= 0)
lambda_peak = lam[np.argmax(radiance.value, axis=0), 0]
lambda_formula = SI.b / T
assert np.all(lambda_peak.isclose(lambda_formula, rel_tol=0, abs_tol=0.02 * si.um))
print('Grid peaks:', lambda_peak)
print('Wien prediction:', lambda_formula.to(lambda_peak.unit))